# A Framework for Bitcoin Bubble Risk Management

## Libraries upload

In [ ]:
pip install lppls

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
#import time
import numpy as np
from sklearn.preprocessing import MinMaxScaler

## Full dataset upload

In [ ]:
df = pd.read_csv('df.csv')
print(df.head())

In [ ]:
df = df.rename(columns={'Close': 'BCP', 'fear_greed_index': 'FGI'})

df['Date'] = pd.to_datetime(df['Date'])

print(df.head())

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['BCP'])
plt.title('Close Price of df Dataset')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.grid(True)
plt.show()

### Seleziono l'intervallo di interesse: giugno 2022- marzo 2026


In [ ]:
df.tail()

In [ ]:
start_date = pd.to_datetime('2022-06-22')
end_date = pd.to_datetime('2026-03-23')

df_filtered = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)].copy()

print("Filtered DataFrame (df_filtered) from November 2018 to January 2023:")
print(df_filtered.head())
print(df_filtered.tail())

In [ ]:
df_filtered = df_filtered.reset_index(drop=True)
print(df_filtered.head())

In [ ]:
df = df_filtered

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['BCP'])
plt.title('Close Price of df Dataset')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.grid(True)
plt.show()

## LPPLS Bubbles Indicator

In [ ]:
import pandas as pd
df = pd.read_csv('dfA.csv')
print(df.head())
print(df.tail())

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

### Nuova versione solo con R2 e VALID FITS

In [ ]:
import numpy as np
import pandas as pd
from lppls import lppls
from tqdm.notebook import tqdm

# ==============================
# PARAMETERS
# ==============================
MAX_WINDOW_SIZE = 210
MIN_WINDOW_SIZE = 30
WINDOW_STEP = 15

N_FITS = 15          # increased from 5
MAX_SEARCHES = 5

MIN_TC_DAYS = 0
MAX_TC_DAYS = 180

M_MIN, M_MAX = 0.01, 0.99
OMEGA_MIN, OMEGA_MAX = 4.0, 17.0

SMOOTH_WINDOW = 25
TC_SMOOTH_WINDOW = 20
NORM_WINDOW = 365

W_R2 = 0.60
W_URGENCY = 0.40

FIT_WEIGHT_SCALE = 10


# ==============================
# HELPER FUNCTIONS
# ==============================
def lppls_function(t, tc, m, w, a, b, c, phi):
    dt = np.maximum(tc - t, 1e-6)
    return a + b * (dt ** m) + c * (dt ** m) * np.cos(w * np.log(dt) + phi)


def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0.0


def urgency_exp(days_to_tc, scale=90):
    """
    Smooth urgency transformation.
    High when tc is close, lower when tc is far.
    """
    if pd.isna(days_to_tc):
        return 0.0

    days_to_tc = max(days_to_tc, 0)
    return float(np.exp(-days_to_tc / scale))


# ==============================
# PREPARE DATA
# ==============================
df = df.copy()

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

if 'log_price' not in df.columns:
    df['log_price'] = np.log(df['BCP'])

if 't' not in df.columns:
    df['t'] = np.arange(len(df))

# ==============================
# OUTPUT COLUMNS
# ==============================
output_cols = [
    'lppls_score_raw',
    'lppls_score',
    'lppls_r2_score',
    'lppls_urgency',
    'lppls_fit_weight',
    'lppls_tc_median',
    'lppls_days_to_tc',
    'lppls_valid_fits'
]

for col in output_cols:
    df[col] = np.nan

df['lppls_score_raw'] = 0.0
df['lppls_score'] = 0.0
df['lppls_r2_score'] = 0.0
df['lppls_urgency'] = 0.0
df['lppls_fit_weight'] = 0.0
df['lppls_valid_fits'] = 0


# ==============================
# MAIN LOOP
# ==============================
print("Starting fixed LPPLS score: R² + smoothed urgency + fit-density weighting")

for i in tqdm(range(MAX_WINDOW_SIZE, len(df)), desc="LPPLS fixed score"):

    current_t = df.loc[i, 't']

    tc_list = []
    r2_list = []

    for window_size in range(MIN_WINDOW_SIZE, MAX_WINDOW_SIZE + 1, WINDOW_STEP):

        window = df.iloc[i - window_size:i]

        observations = np.array([
            window['t'].values,
            window['log_price'].values
        ])

        model = lppls.LPPLS(observations=observations)

        for _ in range(N_FITS):

            try:
                fit_result = model.fit(max_searches=MAX_SEARCHES)

                if not isinstance(fit_result, tuple) or len(fit_result) < 7:
                    continue

                tc, m, w, a, b, c, phi = fit_result[:7]

                days_to_tc = tc - current_t

                # ------------------------------
                # Structural filters
                # ------------------------------
                if days_to_tc < MIN_TC_DAYS:
                    continue

                if b >= 0:
                    continue

                if not (M_MIN < m < M_MAX):
                    continue

                if not (OMEGA_MIN < w < OMEGA_MAX):
                    continue

                # ------------------------------
                # R²
                # ------------------------------
                y_true = window['log_price'].values
                y_pred = lppls_function(
                    window['t'].values,
                    tc, m, w, a, b, c, phi
                )

                r2 = compute_r2(y_true, y_pred)
                r2 = np.clip(r2, 0, 1)

                tc_list.append(tc)
                r2_list.append(r2)

            except Exception:
                continue

    # ==============================
    # DAILY AGGREGATION
    # ==============================
    n_valid = len(tc_list)
    df.loc[i, 'lppls_valid_fits'] = n_valid

    # No hard MIN_VALID_FITS cutoff anymore
    if n_valid == 0:
        continue

    tc_array = np.array(tc_list)
    r2_array = np.array(r2_list)

    r2_score = float(np.mean(r2_array))

    if r2_array.sum() > 0:
        tc_consensus = float(np.average(tc_array, weights=r2_array))
    else:
        tc_consensus = float(np.median(tc_array))

    days_to_tc = tc_consensus - current_t

    urgency = urgency_exp(days_to_tc, scale=MAX_TC_DAYS)

    raw_score = (r2_score ** W_R2) * (urgency ** W_URGENCY)

    # soft confidence penalty instead of hard cutoff
    fit_weight = float(np.tanh(n_valid / FIT_WEIGHT_SCALE))

    score = raw_score * fit_weight

    df.loc[i, 'lppls_score_raw'] = raw_score
    df.loc[i, 'lppls_score'] = score
    df.loc[i, 'lppls_r2_score'] = r2_score
    df.loc[i, 'lppls_urgency'] = urgency
    df.loc[i, 'lppls_fit_weight'] = fit_weight
    df.loc[i, 'lppls_tc_median'] = tc_consensus
    df.loc[i, 'lppls_days_to_tc'] = days_to_tc


# ==============================
# POST-PROCESSING: SMOOTH TC FIRST
# ==============================
df['lppls_tc_median_smooth'] = (
    df['lppls_tc_median']
    .ffill()
    .ewm(span=TC_SMOOTH_WINDOW, adjust=False)
    .mean()
)

df['lppls_days_to_tc_smooth'] = df['lppls_tc_median_smooth'] - df['t']

df['lppls_urgency_smooth_tc'] = df['lppls_days_to_tc_smooth'].apply(
    lambda x: urgency_exp(x, scale=90)
)

# recompute smoother score using smoothed tc urgency
df['lppls_score_tc_smooth'] = (
    (df['lppls_r2_score'] ** W_R2)
    * (df['lppls_urgency_smooth_tc'] ** W_URGENCY)
    * df['lppls_fit_weight']
)

In [ ]:
from google.colab import files

# Save the DataFrame to a CSV file
df.to_csv('dfA.csv', index=False)

# Download the file
files.download('dfA.csv')

In [ ]:
plt.figure(figsize=(14,4))

plt.plot(df['Date'], df['lppls_days_to_tc'])

plt.title('lppls_days_to_tc')
plt.show()

In [ ]:
plt.figure(figsize=(14,4))

plt.plot(df['Date'], df['lppls_urgency'])

plt.title('lppls_urgency')
plt.show()

In [ ]:
plt.figure(figsize=(14,4))

plt.plot(df['Date'], df['lppls_r2_score'])

plt.title('lppls_r2_score')
plt.show()

In [ ]:
plt.figure(figsize=(14,4))

plt.plot(df['Date'], df['lppls_valid_fits'])

plt.title('lppls_valid_fits')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ==============================
# PREP DATA
# ==============================
df_plot = df.copy()

# ==============================
# PLOT
# ==============================
fig, ax1 = plt.subplots(figsize=(15,6))

# --- PRICE ---
ax1.plot(
    df_plot['Date'],
    df_plot['BCP'],
    linewidth=2
)

ax1.set_ylabel(
    'BTC Price',
    fontsize=20
)

ax1.set_xlabel(
    'Date',
    fontsize=20
)

# Bigger tick labels
ax1.tick_params(axis='both', labelsize=20)

# --- SCORE ---
ax2 = ax1.twinx()

ax2.plot(
    df_plot['Date'],
    df_plot['lppls_r2_score'],
    linestyle='--',
    color='orange',
    linewidth=2
)

ax2.set_ylabel(
    'Average R2',
    fontsize=20
)

# Bigger tick labels for second axis
ax2.tick_params(axis='y', labelsize=20)

# --- TITLE ---
plt.title(
    'LPPLS Average R2 Score',
    fontsize=25
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ==============================
# PREP DATA
# ==============================
df_plot = df.copy()


# ==============================
# PLOT
# ==============================
fig, ax1 = plt.subplots(figsize=(15,6))

# --- PRICE ---
ax1.plot(df_plot['Date'], df_plot['BCP'])
ax1.set_ylabel('BTC Price', fontsize=20)
ax1.set_xlabel('Date', fontsize=20)

# --- SCORE ---
ax2 = ax1.twinx()
ax2.plot(
    df_plot['Date'],
    df_plot['lppls_valid_fits'],
    linestyle='--',
    color='green'
)
ax2.set_ylabel('Number of valid fits', fontsize=20)

# --- TITLE ---
plt.title('Number of LPPLS valid fits', fontsize=25)

# Bigger tick labels
ax1.tick_params(axis='both', labelsize=20)

# Bigger tick labels
ax2.tick_params(axis='both', labelsize=20)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define exponents for the new final score
R2_EXPONENT = 0.7
FIT_WEIGHT_EXPONENT = 0.3

# Calculate the new lppls_final_score
# We will use lppls_fit_weight as a proxy for the normalized valid fits
# and handle potential NaNs before calculation
df['lppls_r2_score_filled'] = df['lppls_r2_score'].fillna(0)
df['lppls_valid_fits_filled'] = df['lppls_valid_fits'].fillna(0) / 150

df['lppls_custom_final_score'] = (df['lppls_r2_score_filled'] ** R2_EXPONENT) * \
                               (df['lppls_valid_fits_filled'] ** FIT_WEIGHT_EXPONENT)

# ==============================
# PREP DATA FOR PLOTTING
# ==============================
df_plot = df.copy()

# ==============================
# PLOT THE RESULTS
# ==============================
fig, ax1 = plt.subplots(figsize=(15,6))

# --- PRICE ---
ax1.plot(df_plot['Date'], df_plot['BCP'], color='blue')
ax1.set_ylabel('BTC Price', fontsize=12, color='blue')
ax1.set_xlabel('Date', fontsize=12)
ax1.tick_params(axis='y', labelcolor='blue')

# --- CUSTOM FINAL SCORE ---
ax2 = ax1.twinx()
ax2.plot(
    df_plot['Date'],
    df_plot['lppls_custom_final_score'],
    linestyle='--',
    color='red'
)
ax2.set_ylabel('LPPLS Custom Final Score', fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')

# --- TITLE ---
plt.title('Bitcoin Price vs. LPPLS Custom Final Score', fontsize=14)

fig.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()


### Signal regularization

In [ ]:
import matplotlib.pyplot as plt

SMOOTH_WINDOW = 5

df['lppls_custom_final_score_smooth'] = (
    df['lppls_custom_final_score']
    .ewm(span=SMOOTH_WINDOW, adjust=False)
    .mean()
)

# ==============================
# PREP DATA FOR PLOTTING
# ==============================
df_plot = df.copy()

# ==============================
# PLOT THE RESULTS
# ==============================
fig, ax1 = plt.subplots(figsize=(15,6))

# --- PRICE ---
ax1.plot(df_plot['Date'], df_plot['BCP'], color='blue', label='BTC Price')
ax1.set_ylabel('BTC Price', fontsize=20, color='blue')
ax1.set_xlabel('Date', fontsize=20)
ax1.tick_params(axis='y', labelcolor='blue')

# --- CUSTOM FINAL SCORE ---
ax2 = ax1.twinx()
ax2.plot(
    df_plot['Date'],
    df_plot['lppls_custom_final_score'],
    linestyle=':',
    color='red', alpha=0.5, label='LPPLS Bubble Score (Raw)'
)
ax2.plot(
    df_plot['Date'],
    df_plot['lppls_custom_final_score_smooth'],
    linestyle='--',
    color='red', label='LPPLS Bubble Score (Final)'
)
ax2.set_ylabel('LPPLS Bubble Score', fontsize=18, color='red')
ax2.tick_params(axis='y', labelcolor='red')

# --- TITLE ---
plt.title('Bitcoin Price and LPPLS Bubble Score', fontsize=25)

# Combine legends
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', fontsize=18)

fig.tight_layout() # Adjust layout to prevent labels from overlapping

# Bigger tick labels
ax1.tick_params(axis='both', labelsize=20)

# Bigger tick labels
ax2.tick_params(axis='both', labelsize=20)

plt.show()


## Strategies su DatasetA

In [ ]:
start_date = pd.to_datetime('2023-01-01')
df = df[df['Date'] >= start_date].reset_index(drop=True)
print(df.head())
print(df.tail())

In [ ]:
dfA = df

In [ ]:
df = dfA

In [ ]:
df['bubble_score']  = df['lppls_custom_final_score_smooth']

In [ ]:
FGI_SMOOTH_WINDOW = 10
df['FGI'] = df['FGI'].ewm(span=FGI_SMOOTH_WINDOW, adjust=False).mean()
print('FGI smoothed with a 10-day EWMA.')

In [ ]:
import matplotlib.pyplot as plt

df_plot = df.copy()

fig, ax1 = plt.subplots(figsize=(15,6))

# Plot BCP on the first y-axis
ax1.plot(df_plot['Date'], df_plot['BCP'], color='blue', label='Bitcoin Price')
ax1.set_ylabel('Bitcoin Price', color='blue', fontsize=20)
ax1.tick_params(axis='y', labelcolor='blue')

# Create a second y-axis for FGI
ax2 = ax1.twinx()
ax2.plot(df_plot['Date'], df_plot['FGI'], color='red', linestyle='--', label='Fear & Greed Index')
ax2.set_ylabel('Fear & Greed Index', color='red', fontsize=20)
ax2.tick_params(axis='y', labelcolor='red')

ax1.set_xlabel('Date', fontsize=12)
plt.title('Bitcoin Price and Fear & Greed Index', fontsize=25)

# Add legends from both axes
lines = ax1.get_lines() + ax2.get_lines()
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=20)

plt.grid(True)
plt.tight_layout()

# Bigger tick labels
ax1.tick_params(axis='both', labelsize=19)

# Bigger tick labels
ax2.tick_params(axis='both', labelsize=19)

plt.show()

### Strategia con LPPLS bubble score

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# SIMPLE STRATEGY WITH PERIODIC REBALANCING
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
SCORE_COL = 'bubble_score'

fee = 0.001
REBALANCE_DAYS = 5

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[SCORE_COL] = df[SCORE_COL].ffill().fillna(0).clip(0, 1)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ============================================================
# STRATEGY: PERIODIC REBALANCING
# ============================================================

# Buy & Hold
df['position_bh'] = 1.0

# Compute target exposure
MAX_EXPOSURE = 1
ALPHA = 0.5

df['target_position'] = MAX_EXPOSURE * (1 - df[SCORE_COL]**ALPHA)

# Initialize actual position
positions = []
current_position = df['target_position'].iloc[0]

for i in range(len(df)):

    # rebalance every N days
    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position'].iloc[i]

    positions.append(current_position)

df['position_strategy'] = positions

# ============================================================
# RETURNS
# ============================================================

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    # trades only occur at rebalancing points now
    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee

    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col


df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)
df, ret_strat, equity_strat, trade_strat = apply_strategy(df, 'position_strategy', fee=fee)

# ============================================================
# METRICS
# ============================================================

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_strat, equity_strat, trade_strat, 'position_strategy', 'Rebalanced Strategy (10d)')
])

display(metrics)

# ============================================================
# DRAWDOWNS
# ============================================================

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_strategy'] = compute_drawdown(df[equity_strat])

# ============================================================
# PLOTS
# ============================================================

# Equity
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
plt.plot(df['Date'], df[equity_strat], label='Rebalanced Strategy (10d)', linewidth=2)
plt.title('Equity Curve')
plt.legend()
plt.grid(True)
plt.show()

# Drawdown
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
plt.plot(df['Date'], df['dd_strategy'], label='Rebalanced Strategy (10d)', linewidth=2)
plt.title('Drawdown')
plt.legend()
plt.grid(True)
plt.show()

# Exposure
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df[SCORE_COL], label='Bubble Score')
plt.plot(df['Date'], df['position_strategy'], label='Exposure (10d rebalanced)')
plt.title('Score vs Exposure')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ==============================
# EQUITY CURVE
# ==============================
plt.figure(figsize=(12, 6))

plt.plot(
    df['Date'],
    df[equity_bh],
    label='Buy & Hold',
    linestyle='--',
    linewidth=2
)

plt.plot(
    df['Date'],
    df[equity_strat],
    label='LPPLS Rebalanced Strategy',
    linewidth=3
)

plt.title(
    'Equity Curve',
    fontsize=25
)

plt.xlabel(
    'Date',
    fontsize=20
)

plt.ylabel(
    'Portfolio Value',
    fontsize=20
)

plt.xticks(fontsize=15)
plt.yticks(fontsize=20)

plt.legend(fontsize=18)
plt.grid(True)

plt.tight_layout()
plt.show()


# ==============================
# DRAWDOWN
# ==============================
plt.figure(figsize=(12, 6))

plt.plot(
    df['Date'],
    df['dd_bh'],
    label='Buy & Hold',
    linestyle='--',
    linewidth=2
)

plt.plot(
    df['Date'],
    df['dd_strategy'],
    label='LPPLS Rebalanced Strategy',
    linewidth=3
)

plt.title(
    'Drawdowns',
    fontsize=25
)

plt.xlabel(
    'Date',
    fontsize=20
)

plt.ylabel(
    'Drawdown',
    fontsize=20
)

plt.xticks(fontsize=15)
plt.yticks(fontsize=20)

plt.legend(fontsize=18)
plt.grid(True)

plt.tight_layout()
plt.show()


# ==============================
# SCORE VS EXPOSURE
# ==============================
plt.figure(figsize=(12, 5))

plt.plot(
    df['Date'],
    df[SCORE_COL],
    label='Bubble Score',
    linewidth=2
)

plt.plot(
    df['Date'],
    df['position_strategy'],
    label='Exposure',
    linewidth=3
)

plt.title(
    'Score and Exposure',
    fontsize=25
)

plt.xlabel(
    'Date',
    fontsize=20
)

plt.ylabel(
    'Value',
    fontsize=20
)

plt.xticks(fontsize=15)
plt.yticks(fontsize=20)

plt.legend(fontsize=18)
plt.grid(True)

plt.tight_layout()
plt.show()

### Strategia con FGI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# FGI-BASED STRATEGY (ANALOGOUS TO BUBBLE SCORE)
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
FGI_COL = 'FGI'

fee = 0.001
REBALANCE_DAYS = 5

MAX_EXPOSURE = 1
ALPHA = 1   # same nonlinear mapping used before

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[FGI_COL] = df[FGI_COL].ffill().fillna(50)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ============================================================
# NORMALIZE FGI
# ============================================================

df['fgi_norm'] = (df[FGI_COL] / 100).clip(0, 1)

# ============================================================
# TARGET EXPOSURE (SAME FORMULA)
# ============================================================

df['target_position_fgi'] = MAX_EXPOSURE * (1 - df['fgi_norm']**ALPHA)

# ============================================================
# PERIODIC REBALANCING
# ============================================================

positions = []
current_position = df['target_position_fgi'].iloc[0]

for i in range(len(df)):

    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position_fgi'].iloc[i]

    positions.append(current_position)

df['position_fgi'] = positions

# ============================================================
# RETURNS
# ============================================================

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee

    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col


# Buy & Hold
df['position_bh'] = 1.0
df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)

# 80% Buy & Hold (fair comparison)
df['position_80_bh'] = 0.80
df, ret_80_bh, equity_80_bh, trade_80_bh = apply_strategy(df, 'position_80_bh', fee=fee)

# FGI strategy
df, ret_fgi, equity_fgi, trade_fgi = apply_strategy(df, 'position_fgi', fee=fee)

# ============================================================
# METRICS
# ============================================================

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_80_bh, equity_80_bh, trade_80_bh, 'position_80_bh', '80% Buy & Hold'),
    compute_metrics(df, ret_fgi, equity_fgi, trade_fgi, 'position_fgi', 'FGI Strategy')
])

display(metrics)

# ============================================================
# DRAWDOWNS
# ============================================================

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_80_bh'] = compute_drawdown(df[equity_80_bh])
df['dd_fgi'] = compute_drawdown(df[equity_fgi])

# ============================================================
# PLOTS
# ============================================================

# Equity comparison
plt.figure(figsize=(12,6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')
plt.plot(df['Date'], df[equity_fgi], label='FGI Strategy', linewidth=2)
plt.title('Equity Curve Comparison')
plt.legend()
plt.grid(True)
plt.show()

# Drawdown comparison
plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')
plt.plot(df['Date'], df['dd_fgi'], label='FGI Strategy', linewidth=2)
plt.title('Drawdown Comparison')
plt.legend()
plt.grid(True)
plt.show()

# FGI vs exposure
plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['fgi_norm'], label='FGI (normalized)')
plt.plot(df['Date'], df['position_fgi'], label='FGI Exposure')
plt.title('FGI vs Strategy Exposure')
plt.legend()
plt.grid(True)
plt.show()

### Strategia LPPLS*FGI con fear exit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LPPLS + FGI STRATEGY WITH FEAR EXIT
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
BUBBLE_COL = 'bubble_score'
FGI_COL = 'FGI'

fee = 0.001
REBALANCE_DAYS = 5
MAX_EXPOSURE = 1
ALPHA = 1

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[BUBBLE_COL] = df[BUBBLE_COL].ffill().fillna(0).clip(0, 1)
df[FGI_COL] = df[FGI_COL].ffill().fillna(50)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# SIGNALS
# ------------------------------------------------------------

df['lppls_score'] = df[BUBBLE_COL]
df['fgi_norm'] = (df[FGI_COL] / 100).clip(0, 1)

# Greed/bubble risk: high when both LPPLS and FGI are high
df['bubble_greed_risk'] = (df['lppls_score'] * df['fgi_norm'])**(0.5)

# Fear risk: high when FGI is low
df['fear_risk'] = 1 - df['fgi_norm']

# Final risk score: risk is high either in euphoric bubble or panic/fear
df['risk_score'] = np.maximum(df['bubble_greed_risk'], df['fear_risk']).clip(0, 1)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df['target_position_risk'] = MAX_EXPOSURE * (1 - df['risk_score']**ALPHA)

# ------------------------------------------------------------
# PERIODIC REBALANCING
# ------------------------------------------------------------

positions = []
current_position = df['target_position_risk'].iloc[0]

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position_risk'].iloc[i]
    positions.append(current_position)

df['position_risk'] = positions

# ------------------------------------------------------------
# APPLY STRATEGY
# ------------------------------------------------------------

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee
    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col

# ------------------------------------------------------------
# BENCHMARKS
# ------------------------------------------------------------

df['position_bh'] = 1.0
df['position_80_bh'] = 0.80

df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)
df, ret_80_bh, equity_80_bh, trade_80_bh = apply_strategy(df, 'position_80_bh', fee=fee)
df, ret_risk, equity_risk, trade_risk = apply_strategy(df, 'position_risk', fee=fee)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_80_bh, equity_80_bh, trade_80_bh, 'position_80_bh', '80% Buy & Hold'),
    compute_metrics(df, ret_risk, equity_risk, trade_risk, 'position_risk', 'LPPLS × FGI + Fear Exit')
])

display(metrics)

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_80_bh'] = compute_drawdown(df[equity_80_bh])
df['dd_risk'] = compute_drawdown(df[equity_risk])

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')
plt.plot(df['Date'], df[equity_risk], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Equity Curve Comparison')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')
plt.plot(df['Date'], df['dd_risk'], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Drawdown Comparison')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['bubble_greed_risk'], label='Bubble-Greed Risk')
plt.plot(df['Date'], df['fear_risk'], label='Fear Risk')
plt.plot(df['Date'], df['risk_score'], label='Final Risk Score', linewidth=2)
plt.title('Risk Components')
plt.xlabel('Date')
plt.ylabel('Risk')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['risk_score'], label='Risk Score')
plt.plot(df['Date'], df['position_risk'], label='BTC Exposure')
plt.title('Risk Score and Portfolio Exposure')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

### Stesso con composizione additiva LBS e FGI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LPPLS + FGI STRATEGY WITH FEAR EXIT
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
BUBBLE_COL = 'bubble_score'
FGI_COL = 'FGI'

fee = 0.001
REBALANCE_DAYS = 5
MAX_EXPOSURE = 1
ALPHA = 1

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[BUBBLE_COL] = df[BUBBLE_COL].ffill().fillna(0).clip(0, 1)
df[FGI_COL] = df[FGI_COL].ffill().fillna(50)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# SIGNALS
# ------------------------------------------------------------

df['lppls_score'] = df[BUBBLE_COL]
df['fgi_norm'] = (df[FGI_COL] / 100).clip(0, 1)

# Greed/bubble risk: high when both LPPLS and FGI are high
df['bubble_greed_risk'] = 0.5*df['lppls_score'] + 0.5*df['fgi_norm']

# Fear risk: high when FGI is low
df['fear_risk'] = 1 - df['fgi_norm']

# Final risk score: risk is high either in euphoric bubble or panic/fear
df['risk_score'] = np.maximum(df['bubble_greed_risk'], df['fear_risk']).clip(0, 1)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df['target_position_risk'] = MAX_EXPOSURE * (1 - df['risk_score']**ALPHA)

# ------------------------------------------------------------
# PERIODIC REBALANCING
# ------------------------------------------------------------

positions = []
current_position = df['target_position_risk'].iloc[0]

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position_risk'].iloc[i]
    positions.append(current_position)

df['position_risk'] = positions

# ------------------------------------------------------------
# APPLY STRATEGY
# ------------------------------------------------------------

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee
    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col

# ------------------------------------------------------------
# BENCHMARKS
# ------------------------------------------------------------

df['position_bh'] = 1.0
df['position_80_bh'] = 0.80

df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)
df, ret_80_bh, equity_80_bh, trade_80_bh = apply_strategy(df, 'position_80_bh', fee=fee)
df, ret_risk, equity_risk, trade_risk = apply_strategy(df, 'position_risk', fee=fee)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_80_bh, equity_80_bh, trade_80_bh, 'position_80_bh', '80% Buy & Hold'),
    compute_metrics(df, ret_risk, equity_risk, trade_risk, 'position_risk', 'LPPLS × FGI + Fear Exit')
])

display(metrics)

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_80_bh'] = compute_drawdown(df[equity_80_bh])
df['dd_risk'] = compute_drawdown(df[equity_risk])

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')
plt.plot(df['Date'], df[equity_risk], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Equity Curve Comparison')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')
plt.plot(df['Date'], df['dd_risk'], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Drawdown Comparison')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['bubble_greed_risk'], label='Bubble-Greed Risk')
plt.plot(df['Date'], df['fear_risk'], label='Fear Risk')
plt.plot(df['Date'], df['risk_score'], label='Final Risk Score', linewidth=2)
plt.title('Risk Components')
plt.xlabel('Date')
plt.ylabel('Risk')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['risk_score'], label='Risk Score')
plt.plot(df['Date'], df['position_risk'], label='BTC Exposure')
plt.title('Risk Score and Portfolio Exposure')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

### Volatility Targeting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# ADAPTIVE VOLATILITY TARGETING STRATEGY
# ============================================================

df = df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

PRICE_COL = "BCP"

INITIAL_CAPITAL = 1.0
FEE = 0.001
REBALANCE_DAYS = 5

VOL_WINDOW = 30
TARGET_VOL_WINDOW = 180
TARGET_VOL_PERCENTILE = 0.15
TRADING_DAYS = 365

# ------------------------------------------------------------
# RETURNS
# ------------------------------------------------------------

df["btc_return"] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# REALIZED VOLATILITY
# ------------------------------------------------------------

df["realized_vol"] = (
    df["btc_return"]
    .rolling(VOL_WINDOW)
    .std()
    * np.sqrt(TRADING_DAYS)
)

# ------------------------------------------------------------
# ADAPTIVE TARGET VOLATILITY
# ------------------------------------------------------------

df["target_vol"] = (
    df["realized_vol"]
    .rolling(TARGET_VOL_WINDOW, min_periods=VOL_WINDOW)
    .quantile(TARGET_VOL_PERCENTILE)
)

df["target_vol"] = df["target_vol"].fillna(
    df["realized_vol"]
    .expanding(min_periods=VOL_WINDOW)
    .quantile(TARGET_VOL_PERCENTILE)
)

df["target_vol"] = df["target_vol"].fillna(
    df["realized_vol"].quantile(TARGET_VOL_PERCENTILE)
)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df["vol_target_weight_raw"] = df["target_vol"] / df["realized_vol"]

df["vol_target_weight_raw"] = (
    df["vol_target_weight_raw"]
    .replace([np.inf, -np.inf], np.nan)
    .clip(0, 1)
    .fillna(1.0)
)

# ------------------------------------------------------------
# REBALANCING EVERY 5 DAYS
# ------------------------------------------------------------

df["vol_target_weight"] = np.nan

current_weight = 1.0

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_weight = df.loc[i, "vol_target_weight_raw"]
    df.loc[i, "vol_target_weight"] = current_weight

# ------------------------------------------------------------
# STRATEGY RETURNS WITH TRANSACTION COSTS
# ------------------------------------------------------------

df["vol_weight_lag"] = df["vol_target_weight"].shift(1).fillna(1.0)

df["vol_turnover"] = (
    df["vol_target_weight"] - df["vol_weight_lag"]
).abs()

df["vol_transaction_cost"] = FEE * df["vol_turnover"]

df["strat_vol_return"] = (
    df["vol_weight_lag"] * df["btc_return"]
    - df["vol_transaction_cost"]
)

# ------------------------------------------------------------
# EQUITY CURVES
# ------------------------------------------------------------

df["equity_vol"] = INITIAL_CAPITAL * (1 + df["strat_vol_return"]).cumprod()
df["equity_bh"] = INITIAL_CAPITAL * (1 + df["btc_return"]).cumprod()

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df["dd_vol"] = df["equity_vol"] / df["equity_vol"].cummax() - 1
df["dd_bh"] = df["equity_bh"] / df["equity_bh"].cummax() - 1

# ------------------------------------------------------------
# PERFORMANCE FUNCTION
# ------------------------------------------------------------

def performance_metrics(data, equity_col, return_col, exposure_col=None):
    total_return = data[equity_col].iloc[-1] / data[equity_col].iloc[0] - 1

    n_days = len(data)

    ann_return = (1 + total_return) ** (TRADING_DAYS / n_days) - 1
    ann_vol = data[return_col].std() * np.sqrt(TRADING_DAYS)

    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    max_drawdown = (
        data[equity_col] / data[equity_col].cummax() - 1
    ).min()

    avg_exposure = data[exposure_col].mean() if exposure_col is not None else 1.0

    return {
        "Total Return": total_return,
        "Average Exposure": avg_exposure,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown
    }

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

results = pd.DataFrame({
    "BUY&HOLD": performance_metrics(
        df,
        equity_col="equity_bh",
        return_col="btc_return",
        exposure_col=None
    ),
    "STRAT VOL": performance_metrics(
        df,
        equity_col="equity_vol",
        return_col="strat_vol_return",
        exposure_col="vol_target_weight"
    )
}).T

print("===== PERFORMANCE COMPARISON =====")
display(results)

print(f"\nTarget volatility percentile: {TARGET_VOL_PERCENTILE:.0%}")

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["realized_vol"], label="Realized Volatility")
plt.plot(df["Date"], df["target_vol"], label="Adaptive Target Volatility", linewidth=2)
plt.title("Realized Volatility and Adaptive Target Volatility")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["vol_target_weight"], label="STRAT VOL Exposure")
plt.title("Adaptive Volatility Targeting Exposure")
plt.xlabel("Date")
plt.ylabel("Bitcoin Allocation")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["equity_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["equity_vol"], label="STRAT VOL", linewidth=2)
plt.title("Equity Curve Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["dd_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["dd_vol"], label="STRAT VOL", linewidth=2)
plt.title("Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)
plt.show()

### Trend Following

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# TREND-FOLLOWING STRATEGY
# ============================================================

df = df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

PRICE_COL = "BCP"

INITIAL_CAPITAL = 1.0
FEE = 0.001
REBALANCE_DAYS = 5
TRADING_DAYS = 365

SHORT_MA_WINDOW = 30
LONG_MA_WINDOW = 90
GAMMA = 10

# ------------------------------------------------------------
# RETURNS
# ------------------------------------------------------------

df["btc_return"] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# MOVING AVERAGES AND TREND SIGNAL
# ------------------------------------------------------------

df["ma_short"] = df[PRICE_COL].rolling(SHORT_MA_WINDOW, min_periods=1).mean()
df["ma_long"] = df[PRICE_COL].rolling(LONG_MA_WINDOW, min_periods=1).mean()

df["trend_signal"] = (
    (df["ma_short"] - df["ma_long"]) / df["ma_long"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df["trend_weight_raw"] = 0.5 * (1 + np.tanh(GAMMA * df["trend_signal"]))
df["trend_weight_raw"] = df["trend_weight_raw"].clip(0, 1)

# ------------------------------------------------------------
# REBALANCING EVERY 5 DAYS
# ------------------------------------------------------------

df["trend_weight"] = np.nan
current_weight = 1.0

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_weight = df.loc[i, "trend_weight_raw"]
    df.loc[i, "trend_weight"] = current_weight

# ------------------------------------------------------------
# STRATEGY RETURNS WITH TRANSACTION COSTS
# ------------------------------------------------------------

df["trend_weight_lag"] = df["trend_weight"].shift(1).fillna(1.0)

df["trend_turnover"] = (
    df["trend_weight"] - df["trend_weight_lag"]
).abs()

df["trend_transaction_cost"] = FEE * df["trend_turnover"]

df["strat_trend_return"] = (
    df["trend_weight_lag"] * df["btc_return"]
    - df["trend_transaction_cost"]
)

# ------------------------------------------------------------
# EQUITY CURVES
# ------------------------------------------------------------

df["equity_trend"] = INITIAL_CAPITAL * (1 + df["strat_trend_return"]).cumprod()
df["equity_bh"] = INITIAL_CAPITAL * (1 + df["btc_return"]).cumprod()

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df["dd_trend"] = df["equity_trend"] / df["equity_trend"].cummax() - 1
df["dd_bh"] = df["equity_bh"] / df["equity_bh"].cummax() - 1

# ------------------------------------------------------------
# PERFORMANCE FUNCTION
# ------------------------------------------------------------

def performance_metrics(data, equity_col, return_col, exposure_col=None):
    total_return = data[equity_col].iloc[-1] / data[equity_col].iloc[0] - 1

    n_days = len(data)
    ann_return = (1 + total_return) ** (TRADING_DAYS / n_days) - 1
    ann_vol = data[return_col].std() * np.sqrt(TRADING_DAYS)

    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    max_drawdown = (
        data[equity_col] / data[equity_col].cummax() - 1
    ).min()

    avg_exposure = data[exposure_col].mean() if exposure_col is not None else 1.0

    return {
        "Total Return": total_return,
        "Average Exposure": avg_exposure,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown
    }

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

results = pd.DataFrame({
    "BUY&HOLD": performance_metrics(
        df,
        equity_col="equity_bh",
        return_col="btc_return",
        exposure_col=None
    ),
    "STRAT TREND": performance_metrics(
        df,
        equity_col="equity_trend",
        return_col="strat_trend_return",
        exposure_col="trend_weight"
    )
}).T

print("===== PERFORMANCE COMPARISON =====")
display(results)

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df[PRICE_COL], label="Bitcoin Price")
plt.plot(df["Date"], df["ma_short"], label=f"MA {SHORT_MA_WINDOW}")
plt.plot(df["Date"], df["ma_long"], label=f"MA {LONG_MA_WINDOW}")
plt.title("Bitcoin Price and Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["trend_signal"], label="Trend Signal")
plt.axhline(0, linestyle="--", label="Neutral Trend")
plt.title("Normalized Trend Signal")
plt.xlabel("Date")
plt.ylabel("Trend Signal")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["trend_weight"], label="STRAT TREND Exposure")
plt.title("Trend-Following Exposure")
plt.xlabel("Date")
plt.ylabel("Bitcoin Allocation")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["equity_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["equity_trend"], label="STRAT TREND", linewidth=2)
plt.title("Equity Curve Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["dd_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["dd_trend"], label="STRAT TREND", linewidth=2)
plt.title("Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)
plt.show()

## Strategies su DatasetB

In [ ]:
start_date = pd.to_datetime('2024-07-01')
dfB = df[df['Date'] >= start_date].reset_index(drop=True)
print(df.head())
print(df.tail())

In [ ]:
df = dfB

### Strategia con LPPLS bubble score

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# SIMPLE STRATEGY WITH PERIODIC REBALANCING
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
SCORE_COL = 'bubble_score'

fee = 0.001
REBALANCE_DAYS = 5

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[SCORE_COL] = df[SCORE_COL].ffill().fillna(0).clip(0, 1)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ============================================================
# STRATEGY: PERIODIC REBALANCING
# ============================================================

# Buy & Hold
df['position_bh'] = 1.0

# Compute target exposure
MAX_EXPOSURE = 1
ALPHA = 0.5

df['target_position'] = MAX_EXPOSURE * (1 - df[SCORE_COL]**ALPHA)

# Initialize actual position
positions = []
current_position = df['target_position'].iloc[0]

for i in range(len(df)):

    # rebalance every N days
    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position'].iloc[i]

    positions.append(current_position)

df['position_strategy'] = positions

# ============================================================
# RETURNS
# ============================================================

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    # trades only occur at rebalancing points now
    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee

    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col


df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)
df, ret_strat, equity_strat, trade_strat = apply_strategy(df, 'position_strategy', fee=fee)

# ============================================================
# METRICS
# ============================================================

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_strat, equity_strat, trade_strat, 'position_strategy', 'Rebalanced Strategy (10d)')
])

display(metrics)

# ============================================================
# DRAWDOWNS
# ============================================================

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_strategy'] = compute_drawdown(df[equity_strat])

# ============================================================
# PLOTS
# ============================================================

# Equity
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
plt.plot(df['Date'], df[equity_strat], label='Rebalanced Strategy (10d)', linewidth=2)
plt.title('Equity Curve')
plt.legend()
plt.grid(True)
plt.show()

# Drawdown
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
plt.plot(df['Date'], df['dd_strategy'], label='Rebalanced Strategy (10d)', linewidth=2)
plt.title('Drawdown')
plt.legend()
plt.grid(True)
plt.show()

# Exposure
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df[SCORE_COL], label='Bubble Score')
plt.plot(df['Date'], df['position_strategy'], label='Exposure (10d rebalanced)')
plt.title('Score vs Exposure')
plt.legend()
plt.grid(True)
plt.show()

### Strategia con FGI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# FGI-BASED STRATEGY (ANALOGOUS TO BUBBLE SCORE)
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
FGI_COL = 'FGI'

fee = 0.001
REBALANCE_DAYS = 5

MAX_EXPOSURE = 1
ALPHA = 1   # same nonlinear mapping used before

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[FGI_COL] = df[FGI_COL].ffill().fillna(50)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ============================================================
# NORMALIZE FGI
# ============================================================

df['fgi_norm'] = (df[FGI_COL] / 100).clip(0, 1)

# ============================================================
# TARGET EXPOSURE (SAME FORMULA)
# ============================================================

df['target_position_fgi'] = MAX_EXPOSURE * (1 - df['fgi_norm']**ALPHA)

# ============================================================
# PERIODIC REBALANCING
# ============================================================

positions = []
current_position = df['target_position_fgi'].iloc[0]

for i in range(len(df)):

    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position_fgi'].iloc[i]

    positions.append(current_position)

df['position_fgi'] = positions

# ============================================================
# RETURNS
# ============================================================

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee

    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col


# Buy & Hold
df['position_bh'] = 1.0
df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)

# 80% Buy & Hold (fair comparison)
df['position_80_bh'] = 0.80
df, ret_80_bh, equity_80_bh, trade_80_bh = apply_strategy(df, 'position_80_bh', fee=fee)

# FGI strategy
df, ret_fgi, equity_fgi, trade_fgi = apply_strategy(df, 'position_fgi', fee=fee)

# ============================================================
# METRICS
# ============================================================

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_80_bh, equity_80_bh, trade_80_bh, 'position_80_bh', '80% Buy & Hold'),
    compute_metrics(df, ret_fgi, equity_fgi, trade_fgi, 'position_fgi', 'FGI Strategy')
])

display(metrics)

# ============================================================
# DRAWDOWNS
# ============================================================

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_80_bh'] = compute_drawdown(df[equity_80_bh])
df['dd_fgi'] = compute_drawdown(df[equity_fgi])

# ============================================================
# PLOTS
# ============================================================

# Equity comparison
plt.figure(figsize=(12,6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')
plt.plot(df['Date'], df[equity_fgi], label='FGI Strategy', linewidth=2)
plt.title('Equity Curve Comparison')
plt.legend()
plt.grid(True)
plt.show()

# Drawdown comparison
plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')
plt.plot(df['Date'], df['dd_fgi'], label='FGI Strategy', linewidth=2)
plt.title('Drawdown Comparison')
plt.legend()
plt.grid(True)
plt.show()

# FGI vs exposure
plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['fgi_norm'], label='FGI (normalized)')
plt.plot(df['Date'], df['position_fgi'], label='FGI Exposure')
plt.title('FGI vs Strategy Exposure')
plt.legend()
plt.grid(True)
plt.show()

### Strategia LPPLS*FGI con fear exit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LPPLS + FGI STRATEGY WITH FEAR EXIT
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
BUBBLE_COL = 'bubble_score'
FGI_COL = 'FGI'

fee = 0.001
REBALANCE_DAYS = 5
MAX_EXPOSURE = 1
ALPHA = 1

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[BUBBLE_COL] = df[BUBBLE_COL].ffill().fillna(0).clip(0, 1)
df[FGI_COL] = df[FGI_COL].ffill().fillna(50)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# SIGNALS
# ------------------------------------------------------------

df['lppls_score'] = df[BUBBLE_COL]
df['fgi_norm'] = (df[FGI_COL] / 100).clip(0, 1)

# Greed/bubble risk: high when both LPPLS and FGI are high
df['bubble_greed_risk'] = (df['lppls_score'] * df['fgi_norm'])**(0.5)

# Fear risk: high when FGI is low
df['fear_risk'] = 1 - df['fgi_norm']

# Final risk score: risk is high either in euphoric bubble or panic/fear
df['risk_score'] = np.maximum(df['bubble_greed_risk'], df['fear_risk']).clip(0, 1)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df['target_position_risk'] = MAX_EXPOSURE * (1 - df['risk_score']**ALPHA)

# ------------------------------------------------------------
# PERIODIC REBALANCING
# ------------------------------------------------------------

positions = []
current_position = df['target_position_risk'].iloc[0]

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position_risk'].iloc[i]
    positions.append(current_position)

df['position_risk'] = positions

# ------------------------------------------------------------
# APPLY STRATEGY
# ------------------------------------------------------------

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee
    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col

# ------------------------------------------------------------
# BENCHMARKS
# ------------------------------------------------------------

df['position_bh'] = 1.0
df['position_80_bh'] = 0.80

df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)
df, ret_80_bh, equity_80_bh, trade_80_bh = apply_strategy(df, 'position_80_bh', fee=fee)
df, ret_risk, equity_risk, trade_risk = apply_strategy(df, 'position_risk', fee=fee)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_80_bh, equity_80_bh, trade_80_bh, 'position_80_bh', '80% Buy & Hold'),
    compute_metrics(df, ret_risk, equity_risk, trade_risk, 'position_risk', 'LPPLS × FGI + Fear Exit')
])

display(metrics)

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_80_bh'] = compute_drawdown(df[equity_80_bh])
df['dd_risk'] = compute_drawdown(df[equity_risk])

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')
plt.plot(df['Date'], df[equity_risk], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Equity Curve Comparison')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')
plt.plot(df['Date'], df['dd_risk'], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Drawdown Comparison')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['bubble_greed_risk'], label='Bubble-Greed Risk')
plt.plot(df['Date'], df['fear_risk'], label='Fear Risk')
plt.plot(df['Date'], df['risk_score'], label='Final Risk Score', linewidth=2)
plt.title('Risk Components')
plt.xlabel('Date')
plt.ylabel('Risk')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['risk_score'], label='Risk Score')
plt.plot(df['Date'], df['position_risk'], label='BTC Exposure')
plt.title('Risk Score and Portfolio Exposure')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

# ==============================
# EQUITY CURVE
# ==============================
plt.figure(figsize=(12,6))

plt.plot(
    df['Date'],
    df[equity_bh],
    label='Buy & Hold',
    linestyle='--',
    linewidth=2
)

# plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')

plt.plot(
    df['Date'],
    df[equity_risk],
    label='LPPLS × FGI + Fear Exit',
    linewidth=3
)

plt.title(
    'Equity Curve Comparison',
    fontsize=22
)

plt.xlabel(
    'Date',
    fontsize=18
)

plt.ylabel(
    'Growth of $1',
    fontsize=18
)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend(fontsize=16)
plt.grid(True)

plt.tight_layout()
plt.show()


# ==============================
# DRAWDOWN
# ==============================
plt.figure(figsize=(12,6))

plt.plot(
    df['Date'],
    df['dd_bh'],
    label='Buy & Hold',
    linestyle='--',
    linewidth=2
)

# plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')

plt.plot(
    df['Date'],
    df['dd_risk'],
    label='LPPLS × FGI + Fear Exit',
    linewidth=3
)

plt.title(
    'Drawdowns',
    fontsize=25
)

plt.xlabel(
    'Date',
    fontsize=20
)

plt.ylabel(
    'Drawdown',
    fontsize=20
)

plt.xticks(fontsize=15)
plt.yticks(fontsize=20)

plt.legend(fontsize=18)
plt.grid(True)

plt.tight_layout()
plt.show()


# ==============================
# RISK COMPONENTS
# ==============================
plt.figure(figsize=(12,5))

plt.plot(
    df['Date'],
    df['bubble_greed_risk'],
    label='Bubble-Greed Risk',
    linewidth=2
)

plt.plot(
    df['Date'],
    df['fear_risk'],
    label='Fear Risk',
    linewidth=2
)

plt.plot(
    df['Date'],
    df['risk_score'],
    label='Final Risk Score',
    linewidth=3
)

plt.title(
    'Risk Components',
    fontsize=22
)

plt.xlabel(
    'Date',
    fontsize=18
)

plt.ylabel(
    'Risk',
    fontsize=18
)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend(fontsize=16)
plt.grid(True)

plt.tight_layout()
plt.show()


# ==============================
# RISK SCORE VS EXPOSURE
# ==============================
plt.figure(figsize=(12,5))

plt.plot(
    df['Date'],
    df['risk_score'],
    label='Risk Score',
    linewidth=2
)

plt.plot(
    df['Date'],
    df['position_risk'],
    label='BTC Exposure',
    linewidth=3
)

plt.title(
    'Risk Score and Portfolio Exposure',
    fontsize=22
)

plt.xlabel(
    'Date',
    fontsize=18
)

plt.ylabel(
    'Value',
    fontsize=18
)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend(fontsize=16)
plt.grid(True)

plt.tight_layout()
plt.show()

### Stesso con composizione additiva LBS e FGI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LPPLS + FGI STRATEGY WITH FEAR EXIT
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
BUBBLE_COL = 'bubble_score'
FGI_COL = 'FGI'

fee = 0.001
REBALANCE_DAYS = 5
MAX_EXPOSURE = 1
ALPHA = 1

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[BUBBLE_COL] = df[BUBBLE_COL].ffill().fillna(0).clip(0, 1)
df[FGI_COL] = df[FGI_COL].ffill().fillna(50)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# SIGNALS
# ------------------------------------------------------------

df['lppls_score'] = df[BUBBLE_COL]
df['fgi_norm'] = (df[FGI_COL] / 100).clip(0, 1)

# Greed/bubble risk: high when both LPPLS and FGI are high
df['bubble_greed_risk'] = 0.5*df['lppls_score'] + 0.5*df['fgi_norm']

# Fear risk: high when FGI is low
df['fear_risk'] = 1 - df['fgi_norm']

# Final risk score: risk is high either in euphoric bubble or panic/fear
df['risk_score'] = np.maximum(df['bubble_greed_risk'], df['fear_risk']).clip(0, 1)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df['target_position_risk'] = MAX_EXPOSURE * (1 - df['risk_score']**ALPHA)

# ------------------------------------------------------------
# PERIODIC REBALANCING
# ------------------------------------------------------------

positions = []
current_position = df['target_position_risk'].iloc[0]

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position_risk'].iloc[i]
    positions.append(current_position)

df['position_risk'] = positions

# ------------------------------------------------------------
# APPLY STRATEGY
# ------------------------------------------------------------

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee
    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col

# ------------------------------------------------------------
# BENCHMARKS
# ------------------------------------------------------------

df['position_bh'] = 1.0
df['position_80_bh'] = 0.80

df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)
df, ret_80_bh, equity_80_bh, trade_80_bh = apply_strategy(df, 'position_80_bh', fee=fee)
df, ret_risk, equity_risk, trade_risk = apply_strategy(df, 'position_risk', fee=fee)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_80_bh, equity_80_bh, trade_80_bh, 'position_80_bh', '80% Buy & Hold'),
    compute_metrics(df, ret_risk, equity_risk, trade_risk, 'position_risk', 'LPPLS × FGI + Fear Exit')
])

display(metrics)

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_80_bh'] = compute_drawdown(df[equity_80_bh])
df['dd_risk'] = compute_drawdown(df[equity_risk])

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')
plt.plot(df['Date'], df[equity_risk], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Equity Curve Comparison')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')
plt.plot(df['Date'], df['dd_risk'], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Drawdown Comparison')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['bubble_greed_risk'], label='Bubble-Greed Risk')
plt.plot(df['Date'], df['fear_risk'], label='Fear Risk')
plt.plot(df['Date'], df['risk_score'], label='Final Risk Score', linewidth=2)
plt.title('Risk Components')
plt.xlabel('Date')
plt.ylabel('Risk')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['risk_score'], label='Risk Score')
plt.plot(df['Date'], df['position_risk'], label='BTC Exposure')
plt.title('Risk Score and Portfolio Exposure')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

### Volatility Targeting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# ADAPTIVE VOLATILITY TARGETING STRATEGY
# ============================================================

df = df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

PRICE_COL = "BCP"

INITIAL_CAPITAL = 1.0
FEE = 0.001
REBALANCE_DAYS = 5

VOL_WINDOW = 30
TARGET_VOL_WINDOW = 180
TARGET_VOL_PERCENTILE = 0.15
TRADING_DAYS = 365

# ------------------------------------------------------------
# RETURNS
# ------------------------------------------------------------

df["btc_return"] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# REALIZED VOLATILITY
# ------------------------------------------------------------

df["realized_vol"] = (
    df["btc_return"]
    .rolling(VOL_WINDOW)
    .std()
    * np.sqrt(TRADING_DAYS)
)

# ------------------------------------------------------------
# ADAPTIVE TARGET VOLATILITY
# ------------------------------------------------------------

df["target_vol"] = (
    df["realized_vol"]
    .rolling(TARGET_VOL_WINDOW, min_periods=VOL_WINDOW)
    .quantile(TARGET_VOL_PERCENTILE)
)

df["target_vol"] = df["target_vol"].fillna(
    df["realized_vol"]
    .expanding(min_periods=VOL_WINDOW)
    .quantile(TARGET_VOL_PERCENTILE)
)

df["target_vol"] = df["target_vol"].fillna(
    df["realized_vol"].quantile(TARGET_VOL_PERCENTILE)
)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df["vol_target_weight_raw"] = df["target_vol"] / df["realized_vol"]

df["vol_target_weight_raw"] = (
    df["vol_target_weight_raw"]
    .replace([np.inf, -np.inf], np.nan)
    .clip(0, 1)
    .fillna(1.0)
)

# ------------------------------------------------------------
# REBALANCING EVERY 5 DAYS
# ------------------------------------------------------------

df["vol_target_weight"] = np.nan

current_weight = 1.0

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_weight = df.loc[i, "vol_target_weight_raw"]
    df.loc[i, "vol_target_weight"] = current_weight

# ------------------------------------------------------------
# STRATEGY RETURNS WITH TRANSACTION COSTS
# ------------------------------------------------------------

df["vol_weight_lag"] = df["vol_target_weight"].shift(1).fillna(1.0)

df["vol_turnover"] = (
    df["vol_target_weight"] - df["vol_weight_lag"]
).abs()

df["vol_transaction_cost"] = FEE * df["vol_turnover"]

df["strat_vol_return"] = (
    df["vol_weight_lag"] * df["btc_return"]
    - df["vol_transaction_cost"]
)

# ------------------------------------------------------------
# EQUITY CURVES
# ------------------------------------------------------------

df["equity_vol"] = INITIAL_CAPITAL * (1 + df["strat_vol_return"]).cumprod()
df["equity_bh"] = INITIAL_CAPITAL * (1 + df["btc_return"]).cumprod()

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df["dd_vol"] = df["equity_vol"] / df["equity_vol"].cummax() - 1
df["dd_bh"] = df["equity_bh"] / df["equity_bh"].cummax() - 1

# ------------------------------------------------------------
# PERFORMANCE FUNCTION
# ------------------------------------------------------------

def performance_metrics(data, equity_col, return_col, exposure_col=None):
    total_return = data[equity_col].iloc[-1] / data[equity_col].iloc[0] - 1

    n_days = len(data)

    ann_return = (1 + total_return) ** (TRADING_DAYS / n_days) - 1
    ann_vol = data[return_col].std() * np.sqrt(TRADING_DAYS)

    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    max_drawdown = (
        data[equity_col] / data[equity_col].cummax() - 1
    ).min()

    avg_exposure = data[exposure_col].mean() if exposure_col is not None else 1.0

    return {
        "Total Return": total_return,
        "Average Exposure": avg_exposure,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown
    }

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

results = pd.DataFrame({
    "BUY&HOLD": performance_metrics(
        df,
        equity_col="equity_bh",
        return_col="btc_return",
        exposure_col=None
    ),
    "STRAT VOL": performance_metrics(
        df,
        equity_col="equity_vol",
        return_col="strat_vol_return",
        exposure_col="vol_target_weight"
    )
}).T

print("===== PERFORMANCE COMPARISON =====")
display(results)

print(f"\nTarget volatility percentile: {TARGET_VOL_PERCENTILE:.0%}")

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["realized_vol"], label="Realized Volatility")
plt.plot(df["Date"], df["target_vol"], label="Adaptive Target Volatility", linewidth=2)
plt.title("Realized Volatility and Adaptive Target Volatility")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["vol_target_weight"], label="STRAT VOL Exposure")
plt.title("Adaptive Volatility Targeting Exposure")
plt.xlabel("Date")
plt.ylabel("Bitcoin Allocation")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["equity_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["equity_vol"], label="STRAT VOL", linewidth=2)
plt.title("Equity Curve Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["dd_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["dd_vol"], label="STRAT VOL", linewidth=2)
plt.title("Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)
plt.show()

### Trend Following

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# TREND-FOLLOWING STRATEGY
# ============================================================

df = df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

PRICE_COL = "BCP"

INITIAL_CAPITAL = 1.0
FEE = 0.001
REBALANCE_DAYS = 5
TRADING_DAYS = 365

SHORT_MA_WINDOW = 30
LONG_MA_WINDOW = 90
GAMMA = 10

# ------------------------------------------------------------
# RETURNS
# ------------------------------------------------------------

df["btc_return"] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# MOVING AVERAGES AND TREND SIGNAL
# ------------------------------------------------------------

df["ma_short"] = df[PRICE_COL].rolling(SHORT_MA_WINDOW, min_periods=1).mean()
df["ma_long"] = df[PRICE_COL].rolling(LONG_MA_WINDOW, min_periods=1).mean()

df["trend_signal"] = (
    (df["ma_short"] - df["ma_long"]) / df["ma_long"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df["trend_weight_raw"] = 0.5 * (1 + np.tanh(GAMMA * df["trend_signal"]))
df["trend_weight_raw"] = df["trend_weight_raw"].clip(0, 1)

# ------------------------------------------------------------
# REBALANCING EVERY 5 DAYS
# ------------------------------------------------------------

df["trend_weight"] = np.nan
current_weight = 1.0

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_weight = df.loc[i, "trend_weight_raw"]
    df.loc[i, "trend_weight"] = current_weight

# ------------------------------------------------------------
# STRATEGY RETURNS WITH TRANSACTION COSTS
# ------------------------------------------------------------

df["trend_weight_lag"] = df["trend_weight"].shift(1).fillna(1.0)

df["trend_turnover"] = (
    df["trend_weight"] - df["trend_weight_lag"]
).abs()

df["trend_transaction_cost"] = FEE * df["trend_turnover"]

df["strat_trend_return"] = (
    df["trend_weight_lag"] * df["btc_return"]
    - df["trend_transaction_cost"]
)

# ------------------------------------------------------------
# EQUITY CURVES
# ------------------------------------------------------------

df["equity_trend"] = INITIAL_CAPITAL * (1 + df["strat_trend_return"]).cumprod()
df["equity_bh"] = INITIAL_CAPITAL * (1 + df["btc_return"]).cumprod()

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df["dd_trend"] = df["equity_trend"] / df["equity_trend"].cummax() - 1
df["dd_bh"] = df["equity_bh"] / df["equity_bh"].cummax() - 1

# ------------------------------------------------------------
# PERFORMANCE FUNCTION
# ------------------------------------------------------------

def performance_metrics(data, equity_col, return_col, exposure_col=None):
    total_return = data[equity_col].iloc[-1] / data[equity_col].iloc[0] - 1

    n_days = len(data)
    ann_return = (1 + total_return) ** (TRADING_DAYS / n_days) - 1
    ann_vol = data[return_col].std() * np.sqrt(TRADING_DAYS)

    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    max_drawdown = (
        data[equity_col] / data[equity_col].cummax() - 1
    ).min()

    avg_exposure = data[exposure_col].mean() if exposure_col is not None else 1.0

    return {
        "Total Return": total_return,
        "Average Exposure": avg_exposure,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown
    }

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

results = pd.DataFrame({
    "BUY&HOLD": performance_metrics(
        df,
        equity_col="equity_bh",
        return_col="btc_return",
        exposure_col=None
    ),
    "STRAT TREND": performance_metrics(
        df,
        equity_col="equity_trend",
        return_col="strat_trend_return",
        exposure_col="trend_weight"
    )
}).T

print("===== PERFORMANCE COMPARISON =====")
display(results)

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df[PRICE_COL], label="Bitcoin Price")
plt.plot(df["Date"], df["ma_short"], label=f"MA {SHORT_MA_WINDOW}")
plt.plot(df["Date"], df["ma_long"], label=f"MA {LONG_MA_WINDOW}")
plt.title("Bitcoin Price and Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["trend_signal"], label="Trend Signal")
plt.axhline(0, linestyle="--", label="Neutral Trend")
plt.title("Normalized Trend Signal")
plt.xlabel("Date")
plt.ylabel("Trend Signal")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["trend_weight"], label="STRAT TREND Exposure")
plt.title("Trend-Following Exposure")
plt.xlabel("Date")
plt.ylabel("Bitcoin Allocation")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["equity_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["equity_trend"], label="STRAT TREND", linewidth=2)
plt.title("Equity Curve Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["dd_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["dd_trend"], label="STRAT TREND", linewidth=2)
plt.title("Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)
plt.show()

## Strategies su DatasetC

In [ ]:
start_date = pd.to_datetime('2025-01-01')
dfC = df[df['Date'] >= start_date].reset_index(drop=True)
print(df.head())
print(df.tail())

In [ ]:
df = dfC

### Strategia con LPPLS bubble score

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# SIMPLE STRATEGY WITH PERIODIC REBALANCING
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
SCORE_COL = 'bubble_score'

fee = 0.001
REBALANCE_DAYS = 5

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[SCORE_COL] = df[SCORE_COL].ffill().fillna(0).clip(0, 1)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ============================================================
# STRATEGY: PERIODIC REBALANCING
# ============================================================

# Buy & Hold
df['position_bh'] = 1.0

# Compute target exposure
MAX_EXPOSURE = 1
ALPHA = 0.5

df['target_position'] = MAX_EXPOSURE * (1 - df[SCORE_COL]**ALPHA)

# Initialize actual position
positions = []
current_position = df['target_position'].iloc[0]

for i in range(len(df)):

    # rebalance every N days
    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position'].iloc[i]

    positions.append(current_position)

df['position_strategy'] = positions

# ============================================================
# RETURNS
# ============================================================

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    # trades only occur at rebalancing points now
    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee

    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col


df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)
df, ret_strat, equity_strat, trade_strat = apply_strategy(df, 'position_strategy', fee=fee)

# ============================================================
# METRICS
# ============================================================

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_strat, equity_strat, trade_strat, 'position_strategy', 'Rebalanced Strategy (10d)')
])

display(metrics)

# ============================================================
# DRAWDOWNS
# ============================================================

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_strategy'] = compute_drawdown(df[equity_strat])

# ============================================================
# PLOTS
# ============================================================

# Equity
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
plt.plot(df['Date'], df[equity_strat], label='Rebalanced Strategy (10d)', linewidth=2)
plt.title('Equity Curve')
plt.legend()
plt.grid(True)
plt.show()

# Drawdown
plt.figure(figsize=(12, 6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
plt.plot(df['Date'], df['dd_strategy'], label='Rebalanced Strategy (10d)', linewidth=2)
plt.title('Drawdown')
plt.legend()
plt.grid(True)
plt.show()

# Exposure
plt.figure(figsize=(12, 5))
plt.plot(df['Date'], df[SCORE_COL], label='Bubble Score')
plt.plot(df['Date'], df['position_strategy'], label='Exposure (10d rebalanced)')
plt.title('Score vs Exposure')
plt.legend()
plt.grid(True)
plt.show()

### Strategia con FGI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# FGI-BASED STRATEGY (ANALOGOUS TO BUBBLE SCORE)
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
FGI_COL = 'FGI'

fee = 0.001
REBALANCE_DAYS = 5

MAX_EXPOSURE = 1
ALPHA = 1   # same nonlinear mapping used before

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[FGI_COL] = df[FGI_COL].ffill().fillna(50)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ============================================================
# NORMALIZE FGI
# ============================================================

df['fgi_norm'] = (df[FGI_COL] / 100).clip(0, 1)

# ============================================================
# TARGET EXPOSURE (SAME FORMULA)
# ============================================================

df['target_position_fgi'] = MAX_EXPOSURE * (1 - df['fgi_norm']**ALPHA)

# ============================================================
# PERIODIC REBALANCING
# ============================================================

positions = []
current_position = df['target_position_fgi'].iloc[0]

for i in range(len(df)):

    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position_fgi'].iloc[i]

    positions.append(current_position)

df['position_fgi'] = positions

# ============================================================
# RETURNS
# ============================================================

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee

    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col


# Buy & Hold
df['position_bh'] = 1.0
df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)

# 80% Buy & Hold (fair comparison)
df['position_80_bh'] = 0.80
df, ret_80_bh, equity_80_bh, trade_80_bh = apply_strategy(df, 'position_80_bh', fee=fee)

# FGI strategy
df, ret_fgi, equity_fgi, trade_fgi = apply_strategy(df, 'position_fgi', fee=fee)

# ============================================================
# METRICS
# ============================================================

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_80_bh, equity_80_bh, trade_80_bh, 'position_80_bh', '80% Buy & Hold'),
    compute_metrics(df, ret_fgi, equity_fgi, trade_fgi, 'position_fgi', 'FGI Strategy')
])

display(metrics)

# ============================================================
# DRAWDOWNS
# ============================================================

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_80_bh'] = compute_drawdown(df[equity_80_bh])
df['dd_fgi'] = compute_drawdown(df[equity_fgi])

# ============================================================
# PLOTS
# ============================================================

# Equity comparison
plt.figure(figsize=(12,6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')
plt.plot(df['Date'], df[equity_fgi], label='FGI Strategy', linewidth=2)
plt.title('Equity Curve Comparison')
plt.legend()
plt.grid(True)
plt.show()

# Drawdown comparison
plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')
plt.plot(df['Date'], df['dd_fgi'], label='FGI Strategy', linewidth=2)
plt.title('Drawdown Comparison')
plt.legend()
plt.grid(True)
plt.show()

# FGI vs exposure
plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['fgi_norm'], label='FGI (normalized)')
plt.plot(df['Date'], df['position_fgi'], label='FGI Exposure')
plt.title('FGI vs Strategy Exposure')
plt.legend()
plt.grid(True)
plt.show()

### Strategia LPPLS*FGI con fear exit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LPPLS + FGI STRATEGY WITH FEAR EXIT
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
BUBBLE_COL = 'bubble_score'
FGI_COL = 'FGI'

fee = 0.001
REBALANCE_DAYS = 5
MAX_EXPOSURE = 1
ALPHA = 1

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[BUBBLE_COL] = df[BUBBLE_COL].ffill().fillna(0).clip(0, 1)
df[FGI_COL] = df[FGI_COL].ffill().fillna(50)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# SIGNALS
# ------------------------------------------------------------

df['lppls_score'] = df[BUBBLE_COL]
df['fgi_norm'] = (df[FGI_COL] / 100).clip(0, 1)

# Greed/bubble risk: high when both LPPLS and FGI are high
df['bubble_greed_risk'] = (df['lppls_score'] * df['fgi_norm'])**(0.5)

# Fear risk: high when FGI is low
df['fear_risk'] = 1 - df['fgi_norm']

# Final risk score: risk is high either in euphoric bubble or panic/fear
df['risk_score'] = np.maximum(df['bubble_greed_risk'], df['fear_risk']).clip(0, 1)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df['target_position_risk'] = MAX_EXPOSURE * (1 - df['risk_score']**ALPHA)

# ------------------------------------------------------------
# PERIODIC REBALANCING
# ------------------------------------------------------------

positions = []
current_position = df['target_position_risk'].iloc[0]

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position_risk'].iloc[i]
    positions.append(current_position)

df['position_risk'] = positions

# ------------------------------------------------------------
# APPLY STRATEGY
# ------------------------------------------------------------

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee
    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col

# ------------------------------------------------------------
# BENCHMARKS
# ------------------------------------------------------------

df['position_bh'] = 1.0
df['position_80_bh'] = 0.80

df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)
df, ret_80_bh, equity_80_bh, trade_80_bh = apply_strategy(df, 'position_80_bh', fee=fee)
df, ret_risk, equity_risk, trade_risk = apply_strategy(df, 'position_risk', fee=fee)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_80_bh, equity_80_bh, trade_80_bh, 'position_80_bh', '80% Buy & Hold'),
    compute_metrics(df, ret_risk, equity_risk, trade_risk, 'position_risk', 'LPPLS × FGI + Fear Exit')
])

display(metrics)

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_80_bh'] = compute_drawdown(df[equity_80_bh])
df['dd_risk'] = compute_drawdown(df[equity_risk])

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')
plt.plot(df['Date'], df[equity_risk], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Equity Curve Comparison')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')
plt.plot(df['Date'], df['dd_risk'], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Drawdown Comparison')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['bubble_greed_risk'], label='Bubble-Greed Risk')
plt.plot(df['Date'], df['fear_risk'], label='Fear Risk')
plt.plot(df['Date'], df['risk_score'], label='Final Risk Score', linewidth=2)
plt.title('Risk Components')
plt.xlabel('Date')
plt.ylabel('Risk')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['risk_score'], label='Risk Score')
plt.plot(df['Date'], df['position_risk'], label='BTC Exposure')
plt.title('Risk Score and Portfolio Exposure')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

### Stesso con composizione additiva LBS e FGI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LPPLS + FGI STRATEGY WITH FEAR EXIT
# ============================================================

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

PRICE_COL = 'BCP'
BUBBLE_COL = 'bubble_score'
FGI_COL = 'FGI'

fee = 0.001
REBALANCE_DAYS = 5
MAX_EXPOSURE = 1
ALPHA = 1

# ------------------------------------------------------------
# DATA PREPARATION
# ------------------------------------------------------------

df[PRICE_COL] = df[PRICE_COL].astype(float)
df[BUBBLE_COL] = df[BUBBLE_COL].ffill().fillna(0).clip(0, 1)
df[FGI_COL] = df[FGI_COL].ffill().fillna(50)

df['ret'] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# SIGNALS
# ------------------------------------------------------------

df['lppls_score'] = df[BUBBLE_COL]
df['fgi_norm'] = (df[FGI_COL] / 100).clip(0, 1)

# Greed/bubble risk: high when both LPPLS and FGI are high
df['bubble_greed_risk'] = 0.5*df['lppls_score'] + 0.5*df['fgi_norm']

# Fear risk: high when FGI is low
df['fear_risk'] = 1 - df['fgi_norm']

# Final risk score: risk is high either in euphoric bubble or panic/fear
df['risk_score'] = np.maximum(df['bubble_greed_risk'], df['fear_risk']).clip(0, 1)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df['target_position_risk'] = MAX_EXPOSURE * (1 - df['risk_score']**ALPHA)

# ------------------------------------------------------------
# PERIODIC REBALANCING
# ------------------------------------------------------------

positions = []
current_position = df['target_position_risk'].iloc[0]

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_position = df['target_position_risk'].iloc[i]
    positions.append(current_position)

df['position_risk'] = positions

# ------------------------------------------------------------
# APPLY STRATEGY
# ------------------------------------------------------------

def apply_strategy(df, position_col, ret_col='ret', fee=0.001):
    out = df.copy()

    trade_col = f'trade_{position_col}'
    ret_strategy_col = f'ret_{position_col}'
    equity_col = f'equity_{position_col}'

    out[trade_col] = out[position_col].diff().abs().fillna(0)

    out[ret_strategy_col] = (
        out[position_col].shift(1).fillna(out[position_col]) * out[ret_col]
    )

    out[ret_strategy_col] -= out[trade_col] * fee
    out[equity_col] = (1 + out[ret_strategy_col]).cumprod()

    return out, ret_strategy_col, equity_col, trade_col

# ------------------------------------------------------------
# BENCHMARKS
# ------------------------------------------------------------

df['position_bh'] = 1.0
df['position_80_bh'] = 0.80

df, ret_bh, equity_bh, trade_bh = apply_strategy(df, 'position_bh', fee=fee)
df, ret_80_bh, equity_80_bh, trade_80_bh = apply_strategy(df, 'position_80_bh', fee=fee)
df, ret_risk, equity_risk, trade_risk = apply_strategy(df, 'position_risk', fee=fee)

# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

def compute_drawdown(equity):
    peak = equity.cummax()
    return (equity - peak) / peak


def compute_metrics(df, ret_col, equity_col, trade_col, position_col, name):
    ret = df[ret_col]
    equity = df[equity_col]

    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    ann_return = (1 + total_return) ** (365 / len(df)) - 1
    ann_vol = ret.std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    dd = compute_drawdown(equity)
    max_dd = dd.min()
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    return {
        'Strategy': name,
        'Final Value': equity.iloc[-1],
        'Total Return': total_return,
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Calmar Ratio': calmar,
        'Average Exposure': df[position_col].mean(),
        'Turnover': df[trade_col].sum()
    }


metrics = pd.DataFrame([
    compute_metrics(df, ret_bh, equity_bh, trade_bh, 'position_bh', 'Buy & Hold'),
    compute_metrics(df, ret_80_bh, equity_80_bh, trade_80_bh, 'position_80_bh', '80% Buy & Hold'),
    compute_metrics(df, ret_risk, equity_risk, trade_risk, 'position_risk', 'LPPLS × FGI + Fear Exit')
])

display(metrics)

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df['dd_bh'] = compute_drawdown(df[equity_bh])
df['dd_80_bh'] = compute_drawdown(df[equity_80_bh])
df['dd_risk'] = compute_drawdown(df[equity_risk])

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df[equity_bh], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df[equity_80_bh], label='80% Buy & Hold')
plt.plot(df['Date'], df[equity_risk], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Equity Curve Comparison')
plt.xlabel('Date')
plt.ylabel('Growth of $1')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,6))
plt.plot(df['Date'], df['dd_bh'], label='Buy & Hold', linestyle='--')
#plt.plot(df['Date'], df['dd_80_bh'], label='80% Buy & Hold')
plt.plot(df['Date'], df['dd_risk'], label='LPPLS × FGI + Fear Exit', linewidth=2)
plt.title('Drawdown Comparison')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['bubble_greed_risk'], label='Bubble-Greed Risk')
plt.plot(df['Date'], df['fear_risk'], label='Fear Risk')
plt.plot(df['Date'], df['risk_score'], label='Final Risk Score', linewidth=2)
plt.title('Risk Components')
plt.xlabel('Date')
plt.ylabel('Risk')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12,5))
plt.plot(df['Date'], df['risk_score'], label='Risk Score')
plt.plot(df['Date'], df['position_risk'], label='BTC Exposure')
plt.title('Risk Score and Portfolio Exposure')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

### Volatility Targeting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# ADAPTIVE VOLATILITY TARGETING STRATEGY
# ============================================================

df = df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

PRICE_COL = "BCP"

INITIAL_CAPITAL = 1.0
FEE = 0.001
REBALANCE_DAYS = 5

VOL_WINDOW = 30
TARGET_VOL_WINDOW = 180
TARGET_VOL_PERCENTILE = 0.15
TRADING_DAYS = 365

# ------------------------------------------------------------
# RETURNS
# ------------------------------------------------------------

df["btc_return"] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# REALIZED VOLATILITY
# ------------------------------------------------------------

df["realized_vol"] = (
    df["btc_return"]
    .rolling(VOL_WINDOW)
    .std()
    * np.sqrt(TRADING_DAYS)
)

# ------------------------------------------------------------
# ADAPTIVE TARGET VOLATILITY
# ------------------------------------------------------------

df["target_vol"] = (
    df["realized_vol"]
    .rolling(TARGET_VOL_WINDOW, min_periods=VOL_WINDOW)
    .quantile(TARGET_VOL_PERCENTILE)
)

df["target_vol"] = df["target_vol"].fillna(
    df["realized_vol"]
    .expanding(min_periods=VOL_WINDOW)
    .quantile(TARGET_VOL_PERCENTILE)
)

df["target_vol"] = df["target_vol"].fillna(
    df["realized_vol"].quantile(TARGET_VOL_PERCENTILE)
)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df["vol_target_weight_raw"] = df["target_vol"] / df["realized_vol"]

df["vol_target_weight_raw"] = (
    df["vol_target_weight_raw"]
    .replace([np.inf, -np.inf], np.nan)
    .clip(0, 1)
    .fillna(1.0)
)

# ------------------------------------------------------------
# REBALANCING EVERY 5 DAYS
# ------------------------------------------------------------

df["vol_target_weight"] = np.nan

current_weight = 1.0

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_weight = df.loc[i, "vol_target_weight_raw"]
    df.loc[i, "vol_target_weight"] = current_weight

# ------------------------------------------------------------
# STRATEGY RETURNS WITH TRANSACTION COSTS
# ------------------------------------------------------------

df["vol_weight_lag"] = df["vol_target_weight"].shift(1).fillna(1.0)

df["vol_turnover"] = (
    df["vol_target_weight"] - df["vol_weight_lag"]
).abs()

df["vol_transaction_cost"] = FEE * df["vol_turnover"]

df["strat_vol_return"] = (
    df["vol_weight_lag"] * df["btc_return"]
    - df["vol_transaction_cost"]
)

# ------------------------------------------------------------
# EQUITY CURVES
# ------------------------------------------------------------

df["equity_vol"] = INITIAL_CAPITAL * (1 + df["strat_vol_return"]).cumprod()
df["equity_bh"] = INITIAL_CAPITAL * (1 + df["btc_return"]).cumprod()

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df["dd_vol"] = df["equity_vol"] / df["equity_vol"].cummax() - 1
df["dd_bh"] = df["equity_bh"] / df["equity_bh"].cummax() - 1

# ------------------------------------------------------------
# PERFORMANCE FUNCTION
# ------------------------------------------------------------

def performance_metrics(data, equity_col, return_col, exposure_col=None):
    total_return = data[equity_col].iloc[-1] / data[equity_col].iloc[0] - 1

    n_days = len(data)

    ann_return = (1 + total_return) ** (TRADING_DAYS / n_days) - 1
    ann_vol = data[return_col].std() * np.sqrt(TRADING_DAYS)

    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    max_drawdown = (
        data[equity_col] / data[equity_col].cummax() - 1
    ).min()

    avg_exposure = data[exposure_col].mean() if exposure_col is not None else 1.0

    return {
        "Total Return": total_return,
        "Average Exposure": avg_exposure,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown
    }

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

results = pd.DataFrame({
    "BUY&HOLD": performance_metrics(
        df,
        equity_col="equity_bh",
        return_col="btc_return",
        exposure_col=None
    ),
    "STRAT VOL": performance_metrics(
        df,
        equity_col="equity_vol",
        return_col="strat_vol_return",
        exposure_col="vol_target_weight"
    )
}).T

print("===== PERFORMANCE COMPARISON =====")
display(results)

print(f"\nTarget volatility percentile: {TARGET_VOL_PERCENTILE:.0%}")

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["realized_vol"], label="Realized Volatility")
plt.plot(df["Date"], df["target_vol"], label="Adaptive Target Volatility", linewidth=2)
plt.title("Realized Volatility and Adaptive Target Volatility")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["vol_target_weight"], label="STRAT VOL Exposure")
plt.title("Adaptive Volatility Targeting Exposure")
plt.xlabel("Date")
plt.ylabel("Bitcoin Allocation")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["equity_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["equity_vol"], label="STRAT VOL", linewidth=2)
plt.title("Equity Curve Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["dd_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["dd_vol"], label="STRAT VOL", linewidth=2)
plt.title("Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)
plt.show()

### Trend Following

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# TREND-FOLLOWING STRATEGY
# ============================================================

df = df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

PRICE_COL = "BCP"

INITIAL_CAPITAL = 1.0
FEE = 0.001
REBALANCE_DAYS = 5
TRADING_DAYS = 365

SHORT_MA_WINDOW = 30
LONG_MA_WINDOW = 90
GAMMA = 10

# ------------------------------------------------------------
# RETURNS
# ------------------------------------------------------------

df["btc_return"] = df[PRICE_COL].pct_change().fillna(0)

# ------------------------------------------------------------
# MOVING AVERAGES AND TREND SIGNAL
# ------------------------------------------------------------

df["ma_short"] = df[PRICE_COL].rolling(SHORT_MA_WINDOW, min_periods=1).mean()
df["ma_long"] = df[PRICE_COL].rolling(LONG_MA_WINDOW, min_periods=1).mean()

df["trend_signal"] = (
    (df["ma_short"] - df["ma_long"]) / df["ma_long"]
).replace([np.inf, -np.inf], np.nan).fillna(0)

# ------------------------------------------------------------
# TARGET EXPOSURE
# ------------------------------------------------------------

df["trend_weight_raw"] = 0.5 * (1 + np.tanh(GAMMA * df["trend_signal"]))
df["trend_weight_raw"] = df["trend_weight_raw"].clip(0, 1)

# ------------------------------------------------------------
# REBALANCING EVERY 5 DAYS
# ------------------------------------------------------------

df["trend_weight"] = np.nan
current_weight = 1.0

for i in range(len(df)):
    if i % REBALANCE_DAYS == 0:
        current_weight = df.loc[i, "trend_weight_raw"]
    df.loc[i, "trend_weight"] = current_weight

# ------------------------------------------------------------
# STRATEGY RETURNS WITH TRANSACTION COSTS
# ------------------------------------------------------------

df["trend_weight_lag"] = df["trend_weight"].shift(1).fillna(1.0)

df["trend_turnover"] = (
    df["trend_weight"] - df["trend_weight_lag"]
).abs()

df["trend_transaction_cost"] = FEE * df["trend_turnover"]

df["strat_trend_return"] = (
    df["trend_weight_lag"] * df["btc_return"]
    - df["trend_transaction_cost"]
)

# ------------------------------------------------------------
# EQUITY CURVES
# ------------------------------------------------------------

df["equity_trend"] = INITIAL_CAPITAL * (1 + df["strat_trend_return"]).cumprod()
df["equity_bh"] = INITIAL_CAPITAL * (1 + df["btc_return"]).cumprod()

# ------------------------------------------------------------
# DRAWDOWNS
# ------------------------------------------------------------

df["dd_trend"] = df["equity_trend"] / df["equity_trend"].cummax() - 1
df["dd_bh"] = df["equity_bh"] / df["equity_bh"].cummax() - 1

# ------------------------------------------------------------
# PERFORMANCE FUNCTION
# ------------------------------------------------------------

def performance_metrics(data, equity_col, return_col, exposure_col=None):
    total_return = data[equity_col].iloc[-1] / data[equity_col].iloc[0] - 1

    n_days = len(data)
    ann_return = (1 + total_return) ** (TRADING_DAYS / n_days) - 1
    ann_vol = data[return_col].std() * np.sqrt(TRADING_DAYS)

    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan

    max_drawdown = (
        data[equity_col] / data[equity_col].cummax() - 1
    ).min()

    avg_exposure = data[exposure_col].mean() if exposure_col is not None else 1.0

    return {
        "Total Return": total_return,
        "Average Exposure": avg_exposure,
        "Annualized Return": ann_return,
        "Annualized Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown
    }

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

results = pd.DataFrame({
    "BUY&HOLD": performance_metrics(
        df,
        equity_col="equity_bh",
        return_col="btc_return",
        exposure_col=None
    ),
    "STRAT TREND": performance_metrics(
        df,
        equity_col="equity_trend",
        return_col="strat_trend_return",
        exposure_col="trend_weight"
    )
}).T

print("===== PERFORMANCE COMPARISON =====")
display(results)

# ------------------------------------------------------------
# PLOTS
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df[PRICE_COL], label="Bitcoin Price")
plt.plot(df["Date"], df["ma_short"], label=f"MA {SHORT_MA_WINDOW}")
plt.plot(df["Date"], df["ma_long"], label=f"MA {LONG_MA_WINDOW}")
plt.title("Bitcoin Price and Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["trend_signal"], label="Trend Signal")
plt.axhline(0, linestyle="--", label="Neutral Trend")
plt.title("Normalized Trend Signal")
plt.xlabel("Date")
plt.ylabel("Trend Signal")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["trend_weight"], label="STRAT TREND Exposure")
plt.title("Trend-Following Exposure")
plt.xlabel("Date")
plt.ylabel("Bitcoin Allocation")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["equity_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["equity_trend"], label="STRAT TREND", linewidth=2)
plt.title("Equity Curve Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["dd_bh"], label="BUY&HOLD", linestyle="--")
plt.plot(df["Date"], df["dd_trend"], label="STRAT TREND", linewidth=2)
plt.title("Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)
plt.show()